# CREMA-D Training Template

A starting point for training an emotion classifier on the CREMA-D spectrogram dataset.

**How to use this template:**
- Run cells from top to bottom.
- Edit the **Configuration** cell to set your batch size, learning rate, image size, and number of epochs.
- Replace the model in the **Model Definition** section with your own architecture.
- Rename this file before you start (e.g. `train_resnet50.ipynb`) so it does not conflict with other team members' notebooks.

**Prerequisites:**
- Python environment active (venv or conda — see `README.md`).
- `data/` folder present in this directory (see `README.md` for the download link).
- Run this notebook from the `training/` directory.

This notebook installs its own Python dependencies via `!pip install` cells — no manual package installation required beyond having Jupyter available.

## 0) Python dependencies

In [ ]:
import subprocess, sys

# Core training dependencies
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "torch", "torchvision",
    "matplotlib",
    "scikit-learn",
    "tqdm",
    "numpy",
])

import torch
import torchvision
print(f"torch       {torch.__version__}")
print(f"torchvision {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1) Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

## 2) Configuration

Edit the values in this cell to suit your experiment.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
# Assumes this notebook is run from the training/ directory.
# If not, set DATA_DIR manually, e.g.: DATA_DIR = Path("/absolute/path/to/data")
DATA_DIR = Path.cwd() / "data"
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR  = DATA_DIR / "test"

# ── Image settings ─────────────────────────────────────────────────────────
# Spectrograms are 640×640 grayscale PNGs.
# Resize to 224×224 for compatibility with most pretrained backbones.
# Increase if your GPU allows — larger images retain more detail.
IMAGE_SIZE = 224

# ── Training hyperparameters ───────────────────────────────────────────────
BATCH_SIZE    = 32
NUM_EPOCHS    = 20
LEARNING_RATE = 1e-3
NUM_WORKERS   = 4     # DataLoader workers; set to 0 on Windows if you get errors

# ── Dataset ────────────────────────────────────────────────────────────────
CLASS_NAMES = ["ANG", "DIS", "FEA", "HAP", "NEU", "SAD"]
NUM_CLASSES = len(CLASS_NAMES)

# ── Device ─────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Output ─────────────────────────────────────────────────────────────────
MODEL_SAVE_PATH = Path("model.pth")

print(f"Data dir : {DATA_DIR}")
print(f"Device   : {DEVICE}")
print(f"Classes  : {CLASS_NAMES}")

## 3) Verify data structure

In [ ]:
assert DATA_DIR.exists(), (
    f"data/ not found at {DATA_DIR}.\n"
    "Download it from the shared drive link in training/README.md and extract it here."
)

print(f"{'Split':<8} {'Emotion':<8} {'PNGs':>6} {'WAVs':>6}")
print("-" * 34)
total_png = total_wav = 0
for split, split_dir in [("train", TRAIN_DIR), ("test", TEST_DIR)]:
    for cls in CLASS_NAMES:
        cls_dir = split_dir / cls
        assert cls_dir.exists(), f"Missing expected directory: {cls_dir}"
        n_png = len(list(cls_dir.glob("*.png")))
        n_wav = len(list(cls_dir.glob("*.wav")))
        total_png += n_png
        total_wav += n_wav
        print(f"{split:<8} {cls:<8} {n_png:>6} {n_wav:>6}")

print("-" * 34)
print(f"{'TOTAL':<17} {total_png:>6} {total_wav:>6}")
print("\nData structure OK.")

## 4) Dataset and data loaders

`ImageFolder` loads PNGs from subdirectories named by class label — matching exactly how the data is organised.
The spectrograms are grayscale, so we convert to RGB to work with standard 3-channel model architectures.

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),   # grayscale → 3-channel for standard backbones
    transforms.RandomHorizontalFlip(),              # light augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])

train_dataset = datasets.ImageFolder(root=str(TRAIN_DIR), transform=train_transforms)
test_dataset  = datasets.ImageFolder(root=str(TEST_DIR),  transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train samples : {len(train_dataset)}")
print(f"Test  samples : {len(test_dataset)}")
print(f"Class mapping : {train_dataset.class_to_idx}")

### Visualise a batch

In [ ]:
def denorm(tensor):
    """Reverse the 0.5/0.5 normalisation for display."""
    return (tensor * 0.5 + 0.5).clamp(0, 1)

images, labels = next(iter(train_loader))
n_show = min(8, len(images))

fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3))
for i, ax in enumerate(axes):
    img = denorm(images[i]).permute(1, 2, 0).numpy()
    ax.imshow(img[:, :, 0], cmap="gray")
    ax.set_title(CLASS_NAMES[labels[i]], fontsize=9)
    ax.axis("off")
plt.suptitle("Sample training spectrograms", y=1.02)
plt.tight_layout()
plt.show()

## 5) Model definition

A simple CNN baseline is provided below. Replace this section with your own architecture.

**Swap-in ideas:**
- **Pretrained ResNet / EfficientNet / ViT**: `torchvision.models.resnet50(weights="IMAGENET1K_V1")` — replace the final FC layer with `nn.Linear(in_features, NUM_CLASSES)`.
- **YOLO backbone**: Use `ultralytics` (`pip install ultralytics`) in classification mode: `YOLO('yolov8n-cls.pt')` and point it at the `data/` directory.
- **Audio model (WAV input)**: Load WAV files with `torchaudio`, extract features (MFCC, mel filterbank), and feed into a 1-D CNN or an audio transformer such as `facebook/wav2vec2-base`.

In [ ]:
class SimpleCNN(nn.Module):
    """Minimal CNN baseline — three conv blocks followed by a classifier head."""

    def __init__(self, num_classes: int = 6):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = SimpleCNN(num_classes=NUM_CLASSES).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")
print(model)

## 6) Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

In [ ]:
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}


def run_epoch(loader, training: bool):
    model.train(training)
    total_loss = correct = total = 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for images, labels in tqdm(loader, desc="train" if training else "eval ", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += len(labels)
    return total_loss / total, correct / total


for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    test_loss,  test_acc  = run_epoch(test_loader,  training=False)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f"Epoch {epoch:>3}/{NUM_EPOCHS}  "
          f"train loss {train_loss:.4f}  acc {train_acc:.4f}  "
          f"| test loss {test_loss:.4f}  acc {test_acc:.4f}")

### Training curves

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history["train_loss"], label="Train")
ax1.plot(epochs, history["test_loss"],  label="Test")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.legend()

ax2.plot(epochs, history["train_acc"], label="Train")
ax2.plot(epochs, history["test_acc"],  label="Test")
ax2.set_title("Accuracy")
ax2.set_xlabel("Epoch")
ax2.legend()

plt.suptitle("Training history")
plt.tight_layout()
plt.show()

## 7) Evaluation

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="evaluating"):
        images = images.to(DEVICE)
        preds  = model(images).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

### Confusion matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
plt.colorbar(im, ax=ax)
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES)
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix (test set)")

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.tight_layout()
plt.show()

## 8) Save the model

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names":      CLASS_NAMES,
    "image_size":       IMAGE_SIZE,
    "num_classes":      NUM_CLASSES,
}, MODEL_SAVE_PATH)

print(f"Model saved to {MODEL_SAVE_PATH.resolve()}")

### Loading the saved model (for inference)

```python
checkpoint = torch.load("model.pth", map_location="cpu")
model = SimpleCNN(num_classes=checkpoint["num_classes"])
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
```